# 📝 Git & GitHub
### Exercises & Solutions — 24 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Repository setup & basic workflow (1-4)
- Branching & merging (5-9)
- Inspecting history & diffs (10-13)
- Undoing changes: reset, revert, checkout, stash (14-18)
- Conflict resolution & advanced operations: cherry-pick, rebase, bisect (19-22)
- Remote workflows & .gitignore (23-24)

**Note:** All exercises run live against real, temporary Git repositories created
in this container (`git init`), so command output is genuine.


---


### 1. Initialize a Repository and Make the First Commit

Initialize a fresh repo, create a file, stage it, and make the initial commit — the absolute basics.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "README.md"), "w") as f:
    f.write("# My Project\n")
run("git add README.md", repo)
run('git commit -m "Initial commit"', repo)
print(run("git log --oneline", repo))

### 2. Checking Repository Status at Different Stages

Create a file, check status (untracked), stage it, check status again (staged), commit, check status again (clean).

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "app.py"), "w") as f:
    f.write("print('hello')\n")

print("1. Untracked:")
print(run("git status -s", repo))

run("git add app.py", repo)
print("\n2. Staged:")
print(run("git status -s", repo))

run('git commit -m "Add app.py"', repo)
print("\n3. Clean (nothing to commit):")
print(run("git status -s", repo) or "(empty output = clean)")

### 3. Staging Specific Changes with git add -p style (partial)

Demonstrate staging ONLY specific files (not all changes) when multiple files are modified, using targeted `git add`.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "a.txt"), "w") as f: f.write("a")
with open(os.path.join(repo, "b.txt"), "w") as f: f.write("b")
run("git add a.txt b.txt", repo)
run('git commit -m "add a and b"', repo)

with open(os.path.join(repo, "a.txt"), "w") as f: f.write("a modified")
with open(os.path.join(repo, "b.txt"), "w") as f: f.write("b modified")

run("git add a.txt", repo)   # stage ONLY a.txt, not b.txt
print("Staged (a only):")
print(run("git diff --staged --name-only", repo))
print("\nUnstaged (b only):")
print(run("git diff --name-only", repo))

### 4. Amending the Last Commit

Make a commit, realize you forgot a file, add it, and use `git commit --amend` to fold it into the SAME commit instead of creating a new one.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "main.py"), "w") as f: f.write("def main(): pass")
run("git add main.py", repo)
run('git commit -m "Add main"', repo)
print("Before amend:", run("git log --oneline", repo))

with open(os.path.join(repo, "utils.py"), "w") as f: f.write("def helper(): pass")
run("git add utils.py", repo)
run('git commit --amend --no-edit', repo)

print("\nAfter amend (still ONE commit, now includes both files):")
print(run("git log --oneline", repo))
print(run("git show --stat HEAD", repo))

### 5. Create a Branch, Commit, and List All Branches

Create a feature branch, make a commit on it, and verify the original `main` branch is UNAFFECTED.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v1")
run("git add f.txt", repo)
run('git commit -m "initial"', repo)

run("git checkout -b feature/new-thing", repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v2 - feature change")
run("git add f.txt", repo)
run('git commit -m "feature change"', repo)

print("Branches:", run("git branch", repo))
run("git checkout main", repo)
with open(os.path.join(repo, "f.txt")) as f:
    print("main's f.txt content (should be v1, UNCHANGED):", f.read())

### 6. Fast-Forward Merge

Create a branch with NO divergence from main since branching, merge it back, and observe Git performs a fast-forward (no merge commit needed).

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("base")
run("git add f.txt", repo); run('git commit -m "base"', repo)

run("git checkout -b feature", repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("feature change")
run("git add f.txt", repo); run('git commit -m "feature change"', repo)

run("git checkout main", repo)
result = run("git merge feature", repo)
print(result)
print("\nLog (should be linear, no merge commit):")
print(run("git log --oneline --graph", repo))

### 7. Three-Way Merge (Both Branches Diverged)

Make commits on BOTH main and a feature branch (true divergence), merge, and observe Git creates an actual merge commit with two parents.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("base")
run("git add f.txt", repo); run('git commit -m "base"', repo)

run("git checkout -b feature", repo)
with open(os.path.join(repo, "g.txt"), "w") as f: f.write("feature file")
run("git add g.txt", repo); run('git commit -m "feature commit"', repo)

run("git checkout main", repo)
with open(os.path.join(repo, "h.txt"), "w") as f: f.write("main file")
run("git add h.txt", repo); run('git commit -m "main commit"', repo)   # main also moved forward!

result = run("git merge feature -m 'merge feature into main'", repo)
print(result)
print("\nLog (should show a merge commit with 2 parents):")
print(run("git log --oneline --graph", repo))
print("\nMerge commit parents:", run("git log -1 --format=%P", repo))

### 8. Deleting a Merged Branch Safely vs Unmerged Branch

Show `-d` (safe delete, refuses if unmerged) vs `-D` (force delete) for branch cleanup.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("base")
run("git add f.txt", repo); run('git commit -m "base"', repo)

run("git checkout -b merged-branch", repo)
run("git checkout main", repo)
run("git merge merged-branch", repo)   # nothing to merge, but branch now "is merged" relative to main
print("Delete merged branch (-d):", run("git branch -d merged-branch", repo))

run("git checkout -b unmerged-branch", repo)
with open(os.path.join(repo, "new.txt"), "w") as f: f.write("unmerged work")
run("git add new.txt", repo); run('git commit -m "unmerged commit"', repo)
run("git checkout main", repo)

print("\nTry deleting UNMERGED branch (-d, should fail):")
print(run("git branch -d unmerged-branch", repo))
print("\nForce delete (-D, should succeed):")
print(run("git branch -D unmerged-branch", repo))

### 9. Renaming a Branch

Rename a branch (e.g. fixing a typo in a feature branch name) using `git branch -m`.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("x")
run("git add f.txt", repo); run('git commit -m "init"', repo)
run("git checkout -b featrue/typo-name", repo)

print("Before rename:", run("git branch", repo))
run("git branch -m featrue/typo-name feature/correct-name", repo)
print("\nAfter rename:", run("git branch", repo))

### 10. Viewing Commit History with Custom Formatting

Use `git log --pretty=format:` with custom placeholders to extract just hash, author, and message — useful for scripting/reporting.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
for i in range(3):
    with open(os.path.join(repo, f"f{i}.txt"), "w") as f: f.write(str(i))
    run(f"git add f{i}.txt", repo)
    run(f'git commit -m "commit number {i}"', repo)

print(run('git log --pretty=format:"%h | %an | %s"', repo))

### 11. git diff Between Two Specific Commits

Compare changes between two ARBITRARY commits (not just working-dir vs HEAD), using commit hashes directly.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("version 1\n")
run("git add f.txt", repo); run('git commit -m "v1"', repo)
first_hash = run("git rev-parse HEAD", repo)

with open(os.path.join(repo, "f.txt"), "w") as f: f.write("version 1\nversion 2 added\n")
run("git add f.txt", repo); run('git commit -m "v2"', repo)
second_hash = run("git rev-parse HEAD", repo)

print(run(f"git diff {first_hash} {second_hash}", repo))

### 12. git show to Inspect a Specific Commit's Full Changes

Use `git show <hash>` to view the complete diff AND metadata of one specific commit in isolation.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("original\n")
run("git add f.txt", repo); run('git commit -m "original content"', repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("original\nnew line\n")
run("git add f.txt", repo); run('git commit -m "added a line"', repo)

print(run("git show HEAD", repo))

### 13. git blame to Find Who Last Changed Each Line

Use `git blame` to attribute each line of a file to the commit that last modified it — essential for debugging "who broke this?".

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("line one\n")
run("git add f.txt", repo); run('git commit -m "add line one"', repo)
with open(os.path.join(repo, "f.txt"), "a") as f: f.write("line two\n")
run("git add f.txt", repo); run('git commit -m "add line two"', repo)

print(run("git blame f.txt", repo))

### 14. git checkout -- to Discard Unstaged Changes

Modify a tracked file WITHOUT committing, then discard the change entirely with `git checkout -- <file>` (or modern `git restore`).

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("original\n")
run("git add f.txt", repo); run('git commit -m "original"', repo)

with open(os.path.join(repo, "f.txt"), "w") as f: f.write("accidentally broke this\n")
print("Before discard:", run("git status -s", repo))

run("git checkout -- f.txt", repo)
print("\nAfter discard:", run("git status -s", repo) or "(clean)")
with open(os.path.join(repo, "f.txt")) as f:
    print("File content restored:", f.read().strip())

### 15. git reset to Unstage Without Losing Changes

Stage a change, then use `git reset <file>` to UNSTAGE it (the change itself is preserved, just no longer staged).

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v1")
run("git add f.txt", repo); run('git commit -m "v1"', repo)

with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v2")
run("git add f.txt", repo)
print("Staged:", run("git diff --staged --name-only", repo))

run("git reset f.txt", repo)
print("\nAfter reset - unstaged but still modified:")
print("Staged:", run("git diff --staged --name-only", repo) or "(none)")
print("Modified (unstaged):", run("git diff --name-only", repo))

### 16. git reset --hard to Discard Commits Entirely (Dangerous, Demonstrated Safely)

Make 2 commits, then `reset --hard` back one commit, proving the commit AND its changes are gone from the working tree.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v1")
run("git add f.txt", repo); run('git commit -m "v1"', repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v2 - risky change")
run("git add f.txt", repo); run('git commit -m "v2"', repo)

print("Before reset:", run("git log --oneline", repo))
run("git reset --hard HEAD~1", repo)
print("\nAfter reset --hard HEAD~1:", run("git log --oneline", repo))
with open(os.path.join(repo, "f.txt")) as f:
    print("File content (back to v1):", f.read())

### 17. git revert for Safe Undo on Shared History

Use `git revert` (creates a NEW commit undoing changes) instead of `reset` — the safe choice once a commit has been pushed/shared.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("good content\n")
run("git add f.txt", repo); run('git commit -m "good commit"', repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("BAD content that breaks prod\n")
run("git add f.txt", repo); run('git commit -m "bad commit"', repo)

bad_hash = run("git rev-parse HEAD", repo)
run(f"git revert --no-edit {bad_hash}", repo)

print("History preserved (3 commits now, not 1):")
print(run("git log --oneline", repo))
with open(os.path.join(repo, "f.txt")) as f:
    print("\nContent restored to good state:", f.read())

### 18. git stash for Temporary Work-in-Progress Storage

Stash uncommitted changes, switch context (verify clean working dir), then pop the stash back to resume work.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("base")
run("git add f.txt", repo); run('git commit -m "base"', repo)

with open(os.path.join(repo, "f.txt"), "w") as f: f.write("work in progress, not ready")
print("Before stash:", run("git status -s", repo))

run("git stash push -m 'WIP: experimenting'", repo)
print("\nAfter stash (clean):", run("git status -s", repo) or "(clean)")
print("Stash list:", run("git stash list", repo))

run("git stash pop", repo)
print("\nAfter pop (work restored):", run("git status -s", repo))

### 19. Creating and Resolving a Real Merge Conflict

Force a genuine conflict (two branches editing the same line), confirm it triggers, then manually resolve and commit.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "shared.txt"), "w") as f: f.write("original\n")
run("git add shared.txt", repo); run('git commit -m "original"', repo)

run("git checkout -b branch-a", repo)
with open(os.path.join(repo, "shared.txt"), "w") as f: f.write("change from A\n")
run("git add shared.txt", repo); run('git commit -m "A change"', repo)

run("git checkout main", repo)
run("git checkout -b branch-b", repo)
with open(os.path.join(repo, "shared.txt"), "w") as f: f.write("change from B\n")
run("git add shared.txt", repo); run('git commit -m "B change"', repo)

run("git checkout main", repo)
run("git merge branch-a -m merge-a", repo)
conflict_result = run("git merge branch-b", repo)
print("Conflict triggered:", "CONFLICT" in conflict_result)

with open(os.path.join(repo, "shared.txt"), "w") as f:
    f.write("merged: both A and B incorporated\n")
run("git add shared.txt", repo)
run("git commit --no-edit", repo)
print("\nResolved. Final log:")
print(run("git log --oneline --graph", repo))

### 20. git cherry-pick to Apply a Specific Commit to Another Branch

Make a commit on a feature branch, then `cherry-pick` JUST that one commit onto main WITHOUT merging the whole branch.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("base")
run("git add f.txt", repo); run('git commit -m "base"', repo)

run("git checkout -b feature", repo)
with open(os.path.join(repo, "hotfix.txt"), "w") as f: f.write("urgent fix")
run("git add hotfix.txt", repo); run('git commit -m "URGENT: critical fix"', repo)
fix_hash = run("git rev-parse HEAD", repo)

with open(os.path.join(repo, "other.txt"), "w") as f: f.write("unrelated work-in-progress")
run("git add other.txt", repo); run('git commit -m "WIP unrelated feature"', repo)

run("git checkout main", repo)
run(f"git cherry-pick {fix_hash}", repo)   # ONLY grabs the urgent fix commit

print("main now has the hotfix but NOT the unrelated WIP:")
print(run("git log --oneline", repo))
print("Files on main:", run("ls", repo))

### 21. Interactive Rebase to Squash Multiple Commits (Non-Interactive Simulation)

Simulate squashing 3 small commits into 1 clean commit using `git reset --soft` (a common alternative to interactive rebase for simple cases).

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v1")
run("git add f.txt", repo); run('git commit -m "wip 1"', repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v1v2")
run("git add f.txt", repo); run('git commit -m "wip 2"', repo)
with open(os.path.join(repo, "f.txt"), "w") as f: f.write("v1v2v3 final")
run("git add f.txt", repo); run('git commit -m "wip 3 - actually done now"', repo)

print("Before squash:", run("git log --oneline", repo))

run("git reset --soft HEAD~3", repo)    # un-commit the last 3, but KEEP their combined changes staged
run('git commit -m "Add feature X (squashed from 3 WIP commits)"', repo)

print("\nAfter squash (clean single commit):", run("git log --oneline", repo))

### 22. git bisect to Find a Bug-Introducing Commit (Simulated)

Simulate `git bisect` by manually demonstrating the binary search concept across a commit range to find which commit introduced a bug.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()

# Build a history where commit 5 introduces a "bug" (a specific marker string)
for i in range(1, 9):
    content = f"version {i}\n" + ("BUG_MARKER\n" if i >= 5 else "")
    with open(os.path.join(repo, "f.txt"), "w") as f: f.write(content)
    run("git add f.txt", repo)
    run(f'git commit -m "version {i}"', repo)

def is_buggy(commit_ref):
    run(f"git checkout {commit_ref}", repo)
    with open(os.path.join(repo, "f.txt")) as f:
        return "BUG_MARKER" in f.read()

# Manual binary search across the 8 commits (mirrors what git bisect automates)
all_commits = run("git log --reverse --format=%H", repo).splitlines()
low, high = 0, len(all_commits) - 1
first_bad = None
while low <= high:
    mid = (low + high) // 2
    if is_buggy(all_commits[mid]):
        first_bad = mid
        high = mid - 1
    else:
        low = mid + 1

run("git checkout main", repo)
print(f"First bad commit found at index {first_bad} (out of {len(all_commits)}), "
      f"which corresponds to 'version {first_bad+1}'")
print("This is exactly the search strategy 'git bisect' automates for you")

### 23. Setting Up .gitignore and Verifying It Takes Effect

Create files matching common ignore patterns BEFORE adding `.gitignore`, then add the ignore file and confirm those files vanish from `git status`.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

repo = new_repo()
os.makedirs(os.path.join(repo, "__pycache__"), exist_ok=True)
with open(os.path.join(repo, "__pycache__", "mod.pyc"), "w") as f: f.write("bytecode")
with open(os.path.join(repo, "secrets.env"), "w") as f: f.write("API_KEY=12345")
with open(os.path.join(repo, "main.py"), "w") as f: f.write("print(1)")

print("Before .gitignore:")
print(run("git status -s", repo))

with open(os.path.join(repo, ".gitignore"), "w") as f:
    f.write("__pycache__/\n*.env\n")

print("\nAfter .gitignore (only main.py and .gitignore itself should be untracked):")
print(run("git status -s", repo))

### 24. Simulating a Remote with a Local Bare Repository

Create a LOCAL bare repo (simulating GitHub), clone it, push from one clone, pull into another — a fully real remote workflow without internet access.

In [ ]:
import subprocess, os, tempfile

def run(cmd, cwd):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def new_repo():
    d = tempfile.mkdtemp()
    run("git init -q -b main", d)
    run("git config user.email test@test.com", d)
    run("git config user.name Tester", d)
    return d

bare_repo = tempfile.mkdtemp()
run("git init --bare -q -b main", bare_repo)

clone_a = tempfile.mkdtemp()
run(f"git clone -q {bare_repo} .", clone_a)
run("git config user.email a@a.com", clone_a)
run("git config user.name CloneA", clone_a)

with open(os.path.join(clone_a, "shared.txt"), "w") as f: f.write("from clone A")
run("git add shared.txt", clone_a)
run('git commit -m "Add shared.txt from clone A"', clone_a)
push_result = run("git push origin main", clone_a)
print("Push result:", "main -> main" in push_result or "->" in push_result)

clone_b = tempfile.mkdtemp()
run(f"git clone -q {bare_repo} .", clone_b)
print("\nClone B sees the file pushed by Clone A:", os.path.exists(os.path.join(clone_b, "shared.txt")))
print(run("git log --oneline", clone_b))